<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# Schwan PdM Data Exploration #

### Import the required libraries ###

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns

### Load all the various data ###

In [ ]:
machines = pd.read_csv("../data/schwan/machines.csv")
errors = pd.read_csv("../data/schwan/errors.csv")
failures = pd.read_csv("../data/schwan/failures.csv")
maintenance = pd.read_csv("../data/schwan/maint.csv")
telemetry = pd.read_csv("../data/schwan/telemetry.csv")

telemetry.head()


### Now we need to figure out how to put them together where they can be ###

We'll start with adding the machine info to the telemetry.  Then we'll figure out how to combine the error, failures, and maintenance into that

In [ ]:
merged_telemetry = pd.merge(telemetry, machines, on="machineID")
merged_telemetry['model'] = merged_telemetry['model'].str.replace('model', '')
merged_telemetry

### Let's convert the datetimes to actual datetimes for all read data ###

In [ ]:
merged_telemetry['datetime'] = pd.to_datetime(merged_telemetry['datetime'], format='%m/%d/%Y %I:%M:%S %p')
errors['datetime'] = pd.to_datetime(errors['datetime'], format='%m/%d/%Y %I:%M:%S %p')
failures['datetime'] = pd.to_datetime(failures['datetime'], format='%m/%d/%Y %I:%M:%S %p')
maintenance['datetime'] = pd.to_datetime(maintenance['datetime'], format='%m/%d/%Y %I:%M:%S %p')


### Plot the life of the machines including errors, faults, and maintenance ###

This might give us an idea of how the data can be broken up and what significance the events might have on the life

In [ ]:
def plot_machine_life(machineID, machine_telemetry, machine_errors, machine_failures, machine_maintenance, telemetry=True, errors=False, failures=False, maintenance=False, title="Lifetime of Machine", save_file=None):
    fig = plt.figure(figsize = (32, 12))
    plt.xlim(machine_telemetry['datetime'].min(), machine_telemetry['datetime'].max())
    
    if telemetry:
        plt.plot(machine_telemetry['datetime'],machine_telemetry['vibration'], color='blue', linestyle ='-', label="Vibration")
        plt.plot(machine_telemetry['datetime'],machine_telemetry['rotate'], color='black', linestyle ='-', label="Rotate")
        plt.plot(machine_telemetry['datetime'],machine_telemetry['volt'], color='y', linestyle ='-', label="Volt")
        plt.plot(machine_telemetry['datetime'],machine_telemetry['pressure'], color='grey', linestyle ='-', label="Pressure")
    
    comp_line_dict={
                'comp1':'--',
                'comp2':(0,(5,5)),
                'comp3':(0,(5,10)),
                'comp4':(0,(10,3)),
                'comp5':'-.',
                }
    
    error_line_dict={
                'error1':'--',
                'error2':(0,(5,5)),
                'error3':(0,(5,10)),
                'error4':(0, (10,3)),
                'error5':'-.',
                }
    
    
    if errors:
        for err_index, error in machine_errors.iterrows():
            plt.axvline(x=error['datetime'], color='orange', linestyle=error_line_dict[error['errorID']])
    
    if failures:   
        for fail_index, failure in machine_failures.iterrows():
            
            plt.axvline(x=failure['datetime'], color='red', linestyle =comp_line_dict[failure['failure']])
        
    if maintenance:   
        for maint_index, maint_event in machine_maintenance.iterrows():
            plt.axvline(x=maint_event['datetime'], color='green', linestyle=comp_line_dict[maint_event['comp']])
    
    
    err_legend_lines = [mlines.Line2D([], [], color='orange', linestyle=linestyle) for linestyle in list(error_line_dict.values())]
    err_legend = plt.legend(err_legend_lines, list(error_line_dict.keys()), loc='upper right')
    fail_legend_lines = [mlines.Line2D([], [], color='red', linestyle=linestyle) for linestyle in list(comp_line_dict.values())]
    fail_legend = plt.legend(fail_legend_lines, list(comp_line_dict.keys()), loc='upper right') 
    maint_legend_lines = [mlines.Line2D([], [], color='green', linestyle=linestyle) for linestyle in list(comp_line_dict.values())]
    maint_legend = plt.legend(maint_legend_lines, list(comp_line_dict.keys()), loc='upper right') 
        
    default_handles, default_labels = plt.gca().get_legend_handles_labels()
    plt.legend(default_handles + err_legend.legend_handles + fail_legend.legend_handles + maint_legend.legend_handles, default_labels +  list(error_line_dict.keys()) + list(comp_line_dict.keys()) + list(comp_line_dict.keys()), loc='upper right')
        
    plt.xlabel("DateTime")
    plt.ylabel("Vibration")
    plt.title(title)
    
    plt.show()
    if save_file != None:
        fig.savefig(save_file + '.tiff', dpi=90, format='tiff', bbox_inches='tight')
    return fig

In [ ]:
machine1_telemetry = merged_telemetry[merged_telemetry['machineID']==1].copy()
machine1_errors = errors[errors['machineID']==1]
machine1_failures = failures[failures['machineID']==1]
machine1_maint = maintenance[maintenance['machineID']==1]

print("Telemetry Shape: " + str(machine1_telemetry.shape))
print("Errors Shape: " + str(machine1_errors.shape))
print("Failures Shape: " + str(machine1_failures.shape))
print("Maintenance Shape: " + str(machine1_maint.shape))


In [ ]:

plot_machine_life(1,machine1_telemetry, machine1_errors, machine1_failures, machine1_maint, telemetry=True, errors=True, save_file="../results/schwan_eda/errorLife_1")
plot_machine_life(1,machine1_telemetry, machine1_errors, machine1_failures, machine1_maint, failures=True,save_file="../results/schwan_eda/failureLife_1")
plot_machine_life(1,machine1_telemetry, machine1_errors, machine1_failures, machine1_maint, telemetry=True, maintenance=True,save_file="../results/schwan_eda/maintLife_1")

plot_machine_life(1,machine1_telemetry, machine1_errors, machine1_failures, machine1_maint, errors=True, failures=True, maintenance=True)

machine1_telemetry

Looking at the results of these graphs, there are strong correlations between errors and failures.  There also seem to be indicators in the telemetry leading to failures based on components.  I think the components could be tied to types of telemetry readings.  

In [ ]:
# for index, machine in machines.iterrows():
#     print(machine['machineID'])
#     machineID = int(machine['machineID'])
#     telemetry_mask = merged_telemetry['machineID']==machineID
#     error_mask = errors['machineID']==machineID
#     failure_mask = failures['machineID']==machineID
#     maintenance_mask = maintenance['machineID']==machineID
    
#     machine_telemetry = merged_telemetry[telemetry_mask]
#     machine_errors = errors[error_mask]
#     machine_failures = failures[failure_mask]
#     machine_maintenance = maintenance[maintenance_mask]
    
#     plot_machine_life(machineID,machine_telemetry, machine_errors, machine_failures, machine_maintenance, errors=True, title="Errors over Lifetime of Machine" +str(machineID), save_file="../results/schwan_eda/errorLife_"+str(machineID))
#     plot_machine_life(machineID,machine_telemetry, machine_errors, machine_failures, machine_maintenance, failures=True, title="Failures over Lifetime of Machine" +str(machineID),save_file="../results/schwan_eda/failureLife_"+str(machineID))
#     plot_machine_life(machineID,machine_telemetry, machine_errors, machine_failures, machine_maintenance, maintenance=True,title="Maintenance over Lifetime of Machine" +str(machineID), save_file="../results/schwan_eda/maintLife_"+str(machineID))

### We need to break up the Telemetry into windows that correspond to "Lives" of machines per component ###

This means we have to iterate through the telemetry and where a failure occurred, the RUL of the component that failed is 0.  We have to work backward from there to fill in the RUL.  While we are doing this, we may want to turn the datetime column into cycles number so that it matches our RUL model format.  We will be generating a RUL for each component.  That RUL will go from the time maintenance is performed on the component until a failure occurs on the component.  

Lets do this by machine so we don't get confused.  



In [ ]:
def get_windows_by_component(telemetry, failures, component):
    if telemetry.empty:
        return None
    component_failures = failures[failures['failure']==component]
    
    return_data = pd.DataFrame()
    
    print("Failures for Component " + component)
    print(telemetry.columns)
    print(telemetry.dtypes)
    last_failure_date = telemetry['datetime'].min()
    for index, failure in component_failures.iterrows():
        failure_date = failure['datetime']
        between_failure_telemetry = telemetry[(telemetry['datetime']< failure_date) & (telemetry['datetime']>=last_failure_date)].copy()

        last_failure_date = failure_date
        between_failure_telemetry['cyc_since_last_fail_'+component] = range(1, len(between_failure_telemetry)+1)
        between_failure_telemetry['rul_'+component] = range(len(between_failure_telemetry), 0, -1)
        training_machine_id = str(telemetry.at[0,'machineID']) + '_' + str(index)
        between_failure_telemetry['training_machine_id'] = training_machine_id
        return_data = pd.concat([return_data, between_failure_telemetry], axis=0)
            
    return return_data

In [ ]:
def apply_errors(telemetry, errors):
    error_types = [
        'error1',
        'error2',
        'error3',
        'error4',
        'error5',
    ]
    earliest_date = telemetry['datetime'].min()
    dataframe_to_merge = pd.DataFrame()
    dataframe_to_merge['datetime'] = telemetry['datetime']
    for error_type in error_types:
        filtered_errors = errors[errors['errorID']==error_type]
        err_cyc_col = 'cyc_since_last_' + error_type
        err_date_col = error_type+'_date'
        dataframe_to_merge[err_date_col] = earliest_date - pd.Timedelta(days=1)
        for index, error_event in filtered_errors.iterrows():
            error_date = error_event['datetime']
            dataframe_to_merge.loc[dataframe_to_merge['datetime']>error_date, err_date_col] = error_date
        dataframe_to_merge[err_cyc_col] = (dataframe_to_merge['datetime'] - dataframe_to_merge[err_date_col]).dt.total_seconds()/3600 # to get hours difference (cycles)
            
    telemetry = pd.merge(telemetry, dataframe_to_merge, on='datetime', how='inner')
    return telemetry

In [ ]:
def apply_maintenance(telemetry, maintenance):
    components = [
        'comp1', 
        'comp2', 
        'comp3', 
        'comp4', 
        ]
    
    earliest_date = telemetry['datetime'].min()
    dataframe_to_merge = pd.DataFrame()
    dataframe_to_merge['datetime'] = telemetry['datetime']
    
    for component in components:
        component_maint_events = maintenance[maintenance['comp']==component]
        maint_cyc_col = 'cyc_since_maint_' + component
        maint_date_col = component+'_maint_date'
        dataframe_to_merge[maint_date_col] = earliest_date
        for index, maint_event in component_maint_events.iterrows():
            maint_date = maint_event['datetime']
            dataframe_to_merge.loc[dataframe_to_merge['datetime']>maint_date, maint_date_col] = maint_date
        dataframe_to_merge[maint_cyc_col] = (dataframe_to_merge['datetime'] - dataframe_to_merge[maint_date_col]).dt.total_seconds()/3600 # to get hours difference (cycles)
        
    telemetry = pd.merge(telemetry, dataframe_to_merge, on='datetime', how='inner')
    return telemetry

In [ ]:
def apply_failures(telemetry, failures):
    components = [
        'comp1', 
        'comp2', 
        'comp3', 
        'comp4', 
        ]
    
    earliest_date = telemetry['datetime'].min()
    dataframe_to_merge = pd.DataFrame()
    dataframe_to_merge['datetime'] = telemetry['datetime']
    
    for component in components:
        component_failure_events = failures[failures['failure']==component]
        failure_cyc_col = 'cyc_since_fail_' + component
        failure_date_col = component+'_fail_date'
        dataframe_to_merge[failure_date_col] = earliest_date
        for index, fail_event in component_failure_events.iterrows():
            fail_date = fail_event['datetime']
            dataframe_to_merge.loc[dataframe_to_merge['datetime']>fail_date, failure_date_col] = fail_date
        dataframe_to_merge[failure_cyc_col] = (dataframe_to_merge['datetime'] - dataframe_to_merge[failure_date_col]).dt.total_seconds()/3600 # to get hours difference (cycles)
        
    telemetry = pd.merge(telemetry, dataframe_to_merge, on='datetime', how='inner')
    return telemetry

In [ ]:
def apply_RUL(telemetry):
    components = [
        'comp1', 
        'comp2', 
        'comp3', 
        'comp4', 
        ]
    
    
    for component in components:
        unique_fail_dates = telemetry[component+'_fail_date']
        for fail_date in unique_fail_dates:
            fail_window = telemetry[telemetry[component+'_fail_date'] ==fail_date]
            max_RUL = fail_window['cyc_since_fail_'+component].max()
            telemetry.loc[fail_window.index, component+'_Max_RUL'] = max_RUL
    
        telemetry[component+'_RUL'] = telemetry[component+'_Max_RUL'] - telemetry['cyc_since_fail_'+component]
            
    return telemetry        

In [ ]:
def combine_data(telemetry, errors, failures, maintenance):
    # Simply label the cycles of the engine in sequence
    telemetry['cycle_in_year'] = range(1, len(telemetry)+1)
    telemetry_errors = apply_errors(telemetry, errors)
    telemetry_maint = apply_maintenance(telemetry_errors, maintenance)
    telemetry_failures = apply_failures(telemetry_maint, failures)

    complete_telemetry = apply_RUL(telemetry_failures)

    return complete_telemetry


In [ ]:
machine1_telemetry['datetime'] = pd.to_datetime(machine1_telemetry['datetime'])

machine1_telemetry_complete = combine_data(machine1_telemetry, machine1_errors, machine1_failures, machine1_maint)
machine1_telemetry_complete

In [ ]:
for index, machine in machines.iterrows():
    # print(machine['machineID'])
    machineID = int(machine['machineID'])
    telemetry_mask = merged_telemetry['machineID']==machineID
    error_mask = errors['machineID']==machineID
    failure_mask = failures['machineID']==machineID
    maintenance_mask = maintenance['machineID']==machineID
    
    machine_telemetry = merged_telemetry[telemetry_mask].copy()
    machine_errors = errors[error_mask].copy()
    machine_failures = failures[failure_mask].copy()
    machine_maintenance = maintenance[maintenance_mask].copy()
    
    component_dict = get_component_windows(machine_telemetry, machine_errors, machine_failures, machine_maintenance)
    components = [
        'comp1', 
        'comp2', 
        'comp3', 
        'comp4', 
        ]
    
    for component in components:
        csv_file_path = '../data/schwan/schwan_'+component+'.csv'
        
        component_dict[component].to_csv(csv_file_path, mode='a', header=False, index=False)

### The wrangled data is saved in CSV files now, let's check it and play with it ###

In [ ]:
def open_data(file_path, column_names):
    # Load the file into Pandas dataframe
    wrangled_data = pd.read_csv(file_path, sep=",", header=None, index_col=False, names=column_names)
    wrangled_data['datetime'] = pd.to_datetime(wrangled_data["datetime"])
    return wrangled_data

In [ ]:
def pearson_correlation_matrix(dataframe, col, title):
    corr_result=dataframe.corr(method = 'pearson')
    corr_result = corr_result.stack()
    corr_result = corr_result[((corr_result != 1.0)&(corr_result != -1.0))]
    corr_result = corr_result[((corr_result > 0.0)|(corr_result < -0))].round(2)
    df = pd.DataFrame(corr_result).reset_index()
    temp = df.loc[(df.level_0 == col)]
    cols = []
    for val in temp.level_1.values:
        cols.append(val)
    cols.append(col)
    sns.set(rc={'figure.figsize':(12,8),'ytick.labelsize':(12)})
    dataframe = dataframe[cols]
    correlation_matrix = dataframe.corr(method='pearson').round(2)
    sns.heatmap(data = correlation_matrix, annot = False, cmap = 'coolwarm_r')
    plt.title("Variable Correlation Heatmap "+title+"\n", fontsize = 20)
    plt.show()

In [ ]:
def correlation_visualization(dataframe, col):
    corr_result=dataframe.corr(method = 'pearson')
    corr_result = corr_result.stack()
    corr_result = corr_result[((corr_result != 1.0)&(corr_result != -1.0))]
    corr_result = corr_result[((corr_result > 0)|(corr_result < -0))].round(2)
    df = pd.DataFrame(corr_result).reset_index()
    temp = df.loc[(df.level_0 == col)]
    cols = []
    for val in temp.level_1.values:
        cols.append(val)
    cols.append(col)
    dataframe = dataframe[cols]
    sns.set(rc={'figure.figsize':(12,8),'ytick.labelsize':(12)})
    sns.pairplot(data = dataframe)
    plt.show()

In [ ]:
# Identify the Column Names for the data
column_names = [
    "datetime",
    "machineID",
    "voltage",
    "rotations",
    "pressure",
    "vibration",
    "machine_model",
    "machine_age",
    "cycle",
    "cyc_since_error1",
    "cyc_since_error2",
    "cyc_since_error3",
    "cyc_since_error4",
    "cyc_since_error5",
    "cyc_since_maint",  # Specific to the component.  Other maintenance not in the data (for now)
    "cyc_since_last_fail", # Specific to the component.  Other failure data not in the data (for now)
    "true_RUL",
    "unique_machine_id",
    ]
components = [
    'comp1',
    'comp2',
    'comp3',
    'comp4',
]

for component in components:
    csv_file_path = '../data/schwan/schwan_'+component+'.csv'
    wrangled_data = open_data(csv_file_path, column_names)

    pearson_correlation_matrix(wrangled_data, 'true_RUL', component.replace('comp', 'Component ') )
    
    # correlation_visualization(wrangled_data, 'true_RUL')

wrangled_data
